In [1]:
!pip -q install -U transformers accelerate peft bitsandbytes sentencepiece protobuf


# AI vs Human-Written Text Classification
### Neuroxlab Datathon #01

**A Multi-Model Ensemble Learning Framework for Detecting AI-Generated Text**

This notebook presents an end-to-end machine learning pipeline for distinguishing human-written text from AI-generated text. It combines lexical, linguistic, statistical, and transformer-based representations with cross-validation, out-of-fold prediction, ensemble learning, and optimized decision thresholds.

**Classification Labels**
- `0` — Human-written
- `1` — AI-generated

---

## Table of Contents

| No. | Section | Description |
|---|---|---|
| 01 | [Environment Setup](#01-environment-setup) | Dependencies and runtime configuration |
| 02 | [Data Loading and Preparation](#02-data-loading-and-preparation) | Dataset loading, labels, folds, and evaluation utilities |
| 03 | [Lexical Feature Engineering](#03-lexical-feature-engineering) | Word-level and character-level TF-IDF |
| 04 | [Linguistic and Statistical Feature Extraction](#04-linguistic-and-statistical-feature-extraction) | GPT-2 language-model features, stylometry, POS, and syntax |
| 05 | [Classical Machine Learning Models](#05-classical-machine-learning-models) | CatBoost and LightGBM |
| 06 | [Transformer Architecture and Training Framework](#06-transformer-architecture-and-training-framework) | Dataset, dynamic padding, attention pooling, and multi-sample dropout |
| 07 | [Transformer Tokenizer Verification](#07-transformer-tokenizer-verification) | DeBERTa and ModernBERT tokenizer checks |
| 08 | [Transformer-Based Model Training](#08-transformer-based-model-training) | DeBERTa-v3-large and ModernBERT-large |
| 09 | [Qwen 2.5 7B Fine-Tuning with QLoRA](#09-qwen-25-7b-fine-tuning-with-qlora) | Quantization, LoRA, and out-of-fold predictions |
| 10 | [Out-of-Fold Prediction and Ensemble Learning](#10-out-of-fold-prediction-and-ensemble-learning) | Model stacking and non-negative logit blending |
| 11 | [Threshold Optimization and Error Analysis](#11-threshold-optimization-and-error-analysis) | Bootstrap-based threshold selection and error auditing |
| 12 | [Final Prediction and Submission Generation](#12-final-prediction-and-submission-generation) | Generate competition predictions |
| 13 | [Submission Validation](#13-submission-validation) | Verify submission structure and integrity |

---

**Evaluation Metric:** F1-Score

**Validation Strategy:** Stratified 10-Fold Cross-Validation

**Ensemble Strategy:** Regularized Logistic Regression Stacking and Non-Negative Logit Blending


# 02. Data Loading and Preparation
### Dataset Inspection, Label Encoding, Stratified Folds, and Evaluation Setup
# 03. Lexical Feature Engineering
### Word-Level and Character-Level TF-IDF Representations

In [3]:
import os
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc, re, glob, math, json, time, string, random, shutil, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import transformers
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score, log_loss

# 1. Auto-detect the folder inside /kaggle/input/ containing train.csv & test.csv
train_matches = glob.glob("/kaggle/input/**/train.csv", recursive=True)
assert len(train_matches) > 0, "train.csv not found! Attach the competition dataset via '+ Add Input'."
DATA_DIR = os.path.dirname(train_matches[0])
print(f"Found competition data in: {DATA_DIR}")
print("Files:", os.listdir(DATA_DIR))

# 2. Writable output folder in Kaggle
OUT_DIR = "/kaggle/working/oof_and_models_10fold"
os.makedirs(OUT_DIR, exist_ok=True)

# (Optional) If you uploaded previous Colab OOF files as a Kaggle Dataset, copy them in to resume
for prev_file in glob.glob("/kaggle/input/**/*.npy", recursive=True) + \
                 glob.glob("/kaggle/input/**/*.json", recursive=True) + \
                 glob.glob("/kaggle/input/**/*.parquet", recursive=True):
    dst = os.path.join(OUT_DIR, os.path.basename(prev_file))
    if not os.path.exists(dst):
        shutil.copy(prev_file, dst)

# ======================== CONFIG ========================
N_FOLDS       = 10
SEED          = 42
MAX_LEN       = 128
DEBERTA_SEEDS = [42, 2026]
RUN_QLORA_7B  = True
# ========================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
HAS_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0

def dtype_kwarg(dt):
    major = int(transformers.__version__.split(".")[0])
    return {"dtype": dt} if major >= 5 else {"torch_dtype": dt}

print(f"GPU: {GPU_NAME} ({VRAM_GB:.1f} GB) x{torch.cuda.device_count()} | bf16: {HAS_BF16} | transformers: {transformers.__version__}")

# Load datasets
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

if train["label"].dtype == object:
    train["target"] = (train["label"].str.strip().str.lower() == "ai").astype(int)
else:
    train["target"] = train["label"].astype(int)

folds_path = os.path.join(OUT_DIR, "train_folds.csv")
if os.path.exists(folds_path):
    train = pd.read_csv(folds_path)
    print(f"Loaded existing 10-fold split from {folds_path}")
else:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    train["fold"] = -1
    for fold, (_, val_idx) in enumerate(skf.split(train, train["target"])):
        train.loc[val_idx, "fold"] = fold
    train.to_csv(folds_path, index=False)
    print(f"Created 10-fold split -> {folds_path}")

y = train["target"].values
folds = train["fold"].values
print(f"Train: {train.shape} (AI ratio={y.mean():.4f}) | Test: {test.shape}")

def evaluate_predictions(name, probs, y_true=y):
    auc = roc_auc_score(y_true, probs)
    ll  = log_loss(y_true, np.clip(probs, 1e-6, 1 - 1e-6))
    f1_05 = f1_score(y_true, (probs >= 0.5).astype(int))
    ths = np.linspace(0.25, 0.75, 101)
    best_th, best_f1 = max([(t, f1_score(y_true, (probs >= t).astype(int))) for t in ths], key=lambda x: x[1])
    print(f"[{name:26s}] AUC: {auc:.5f} | LogLoss: {ll:.5f} | F1@0.50: {f1_05:.5f} | BestF1: {best_f1:.5f} (th={best_th:.3f})")
    return auc, best_f1

Found competition data in: /kaggle/input/datasets/problemthinkerbox/ai-vs-human-text
Files: ['test.csv', 'train.csv']
GPU: Tesla T4 (14.6 GB) x2 | bf16: True | transformers: 5.24.0
Created 10-fold split -> /kaggle/working/oof_and_models_10fold/train_folds.csv
Train: (1039, 9) (AI ratio=0.4986) | Test: (260, 2)


# 04. Linguistic and Statistical Feature Extraction
### Language Model Memorization, Stylometry, Part-of-Speech, and Syntactic Features

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from scipy.sparse import hstack

oof_tfidf_lr  = np.zeros(len(train))
test_tfidf_lr = np.zeros(len(test))
oof_tfidf_svc  = np.zeros(len(train))
test_tfidf_svc = np.zeros(len(test))

for fold in range(N_FOLDS):
    tr_idx = np.where(folds != fold)[0]
    va_idx = np.where(folds == fold)[0]

    tr_txt = train["text"].iloc[tr_idx]
    va_txt = train["text"].iloc[va_idx]
    te_txt = test["text"]
    y_tr   = y[tr_idx]

    vec_w = TfidfVectorizer(
        ngram_range=(1, 3), sublinear_tf=True, min_df=2,
        lowercase=False, token_pattern=r"(?u)\b\w+(?:['’]\w+)?\b|[.,!?;:—…]"
    )
    vec_c = TfidfVectorizer(
        analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True,
        min_df=3, lowercase=False
    )

    X_tr = hstack([vec_w.fit_transform(tr_txt), vec_c.fit_transform(tr_txt)])
    X_va = hstack([vec_w.transform(va_txt), vec_c.transform(va_txt)])
    X_te = hstack([vec_w.transform(te_txt), vec_c.transform(te_txt)])

    lr = LogisticRegression(C=3.0, max_iter=2000)
    lr.fit(X_tr, y_tr)
    oof_tfidf_lr[va_idx] = lr.predict_proba(X_va)[:, 1]
    test_tfidf_lr += lr.predict_proba(X_te)[:, 1] / N_FOLDS

    svc = CalibratedClassifierCV(LinearSVC(C=0.5, max_iter=3000), cv=5)
    svc.fit(X_tr, y_tr)
    oof_tfidf_svc[va_idx] = svc.predict_proba(X_va)[:, 1]
    test_tfidf_svc += svc.predict_proba(X_te)[:, 1] / N_FOLDS

np.save(os.path.join(OUT_DIR, "oof_tfidf_lr.npy"), oof_tfidf_lr)
np.save(os.path.join(OUT_DIR, "test_tfidf_lr.npy"), test_tfidf_lr)
np.save(os.path.join(OUT_DIR, "oof_tfidf_svc.npy"), oof_tfidf_svc)
np.save(os.path.join(OUT_DIR, "test_tfidf_svc.npy"), test_tfidf_svc)

evaluate_predictions("Stream A1: TFIDF-LogReg", oof_tfidf_lr)
evaluate_predictions("Stream A2: TFIDF-LinearSVC", oof_tfidf_svc)

[Stream A1: TFIDF-LogReg   ] AUC: 0.99027 | LogLoss: 0.15586 | F1@0.50: 0.96538 | BestF1: 0.96724 (th=0.505)
[Stream A2: TFIDF-LinearSVC] AUC: 0.99028 | LogLoss: 0.10883 | F1@0.50: 0.96625 | BestF1: 0.96638 (th=0.350)


(np.float64(0.9902807935437493), 0.9663784822286263)

# 05. Classical Machine Learning Models
### CatBoost and LightGBM with Stratified Out-of-Fold Predictions

In [5]:
import spacy
from transformers import AutoTokenizer, AutoModelForCausalLM
from catboost import CatBoostClassifier
import lightgbm as lgb

feat_cache_tr = os.path.join(OUT_DIR, "X_tab_train.parquet")
feat_cache_te = os.path.join(OUT_DIR, "X_tab_test.parquet")

def extract_lm_memorization_features(texts, model_name="gpt2-large"):
    print(f"Extracting frozen LM memorization features with {model_name}...")
    tok = AutoTokenizer.from_pretrained(model_name)
    lm_dtype = torch.bfloat16 if HAS_BF16 else torch.float16
    lm = AutoModelForCausalLM.from_pretrained(model_name, **dtype_kwarg(lm_dtype)).to(DEVICE)
    lm.eval()

    rows = []
    with torch.no_grad():
        for s in texts:
            enc = tok(s, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(DEVICE)
            ids = enc["input_ids"]
            if ids.shape[1] < 2:
                rows.append({k: 0.0 for k in [
                    "lm_mean_nll", "lm_std_nll", "lm_min_nll", "lm_max_nll", "lm_median_nll",
                    "lm_first_half_nll", "lm_second_half_nll", "lm_half_ratio", "lm_nll_slope",
                    "lm_top1_frac", "lm_top10_frac", "lm_top100_frac", "lm_mean_log_rank",
                    "lm_last3_nll", "lm_tok_count"
                ]})
                continue

            logits = lm(ids).logits[0, :-1, :].float()  # (T-1, V)
            targets = ids[0, 1:]                         # (T-1,)
            log_probs = torch.log_softmax(logits, dim=-1)
            nll = -log_probs[torch.arange(len(targets), device=DEVICE), targets]
            target_logits = logits[torch.arange(len(targets), device=DEVICE), targets].unsqueeze(1)
            ranks = (logits > target_logits).sum(dim=-1).float()  # 0-indexed rank

            nll_np = nll.cpu().numpy()
            ranks_np = ranks.cpu().numpy()
            T = len(nll_np)
            mid = max(1, T // 2)
            first_h = float(np.mean(nll_np[:mid]))
            second_h = float(np.mean(nll_np[mid:])) if T > 1 else first_h
            slope = float(np.polyfit(np.arange(T), nll_np, 1)[0]) if T >= 3 else 0.0

            rows.append({
                "lm_mean_nll": float(np.mean(nll_np)),
                "lm_std_nll": float(np.std(nll_np)),
                "lm_min_nll": float(np.min(nll_np)),
                "lm_max_nll": float(np.max(nll_np)),
                "lm_median_nll": float(np.median(nll_np)),
                "lm_first_half_nll": first_h,
                "lm_second_half_nll": second_h,
                "lm_half_ratio": second_h / (first_h + 1e-4),
                "lm_nll_slope": slope,
                "lm_top1_frac": float(np.mean(ranks_np < 1)),
                "lm_top10_frac": float(np.mean(ranks_np < 10)),
                "lm_top100_frac": float(np.mean(ranks_np < 100)),
                "lm_mean_log_rank": float(np.mean(np.log1p(ranks_np))),
                "lm_last3_nll": float(np.mean(nll_np[-3:])),
                "lm_tok_count": float(T),
            })

    del lm, tok
    gc.collect()
    torch.cuda.empty_cache()
    return pd.DataFrame(rows)

def extract_stylometry_and_pos(texts):
    nlp = spacy.load("en_core_web_sm", disable=["ner"])
    tech_re = re.compile(
        r"\b(?:ai|robot|drone|algorithm|chatbot|smart|sensor|neural|model|satellite|autonomous|"
        r"app|software|digital|virtual|quantum|data|system|automated|machine|device|network|"
        r"cloud|server|battery|screen|gps|printer|email|notification|biometric|blockchain|"
        r"hologram|cyber|code|program|computer|phone|laptop|tablet|camera|speaker|thermostat|"
        r"interface|vr|ar|laser|scanner)\b", re.I
    )
    pron12_re = re.compile(
        r"\b(?:i|me|my|mine|myself|you|your|yours|yourself|we|us|our|ours|"
        r"i'm|i’m|i've|i’ve|i'd|i’d|i'll|i’ll|you're|you’re|you've|you’ve|"
        r"you'll|you’ll|we're|we’re|don't|don’t|can't|can’t|won't|won’t)\b", re.I
    )
    pron3_re = re.compile(r"\b(?:he|she|his|her|him|himself|herself|its|itself|their|them)\b", re.I)
    ed_re    = re.compile(r"\b[a-z]{3,}ed\b", re.I)
    ing_re   = re.compile(r"\b[a-z]{3,}ing\b", re.I)
    ly_re    = re.compile(r"\b[a-z]{3,}ly\b", re.I)
    modal_re = re.compile(r"\b(?:can|cannot|can't|can’t|will|won't|won’t|shall|should|must|may|might|would|could|never|always|nothing|everything|everyone|nobody)\b", re.I)
    scene_re = re.compile(r"\b(?:across|through|into|onto|under|over|beside|behind|outside|inside|along|toward|towards|overhead|beneath)\b", re.I)

    rows = []
    for s, doc in zip(texts, nlp.pipe(texts, batch_size=128)):
        words = s.split()
        wc = len(words)
        cc = len(s)
        awl = float(np.mean([len(w) for w in words])) if wc else 0.0
        awl_clean = float(np.mean([len(w.strip(string.punctuation + "’‘“”—…")) for w in words])) if wc else 0.0
        punct_cnt = sum(1 for c in s if c in string.punctuation)
        pdens = punct_cnt / wc if wc else 0.0

        lower_clean = [re.sub(r"[^a-z]", "", w.lower()) for w in words]
        lower_clean = [w for w in lower_clean if w]
        ttr = len(set(lower_clean)) / len(lower_clean) if lower_clean else 1.0

        # Fine-grained Penn Treebank POS tags from spaCy
        tags = [tok.tag_ for tok in doc]
        ntok = max(1, len(tags))
        vbd_cnt = sum(t == "VBD" for t in tags)     # Active past-tense verb (strong AI signal)
        vbn_cnt = sum(t == "VBN" for t in tags)     # Past participle (common in Human quotes)
        vbp_vbz = sum(t in ("VBP", "VBZ") for t in tags)
        vb_cnt  = sum(t == "VB" for t in tags)
        prp_cnt = sum(t in ("PRP", "PRP$") for t in tags)
        dt_cnt  = sum(t == "DT" for t in tags)
        md_cnt  = sum(t == "MD" for t in tags)
        jj_cnt  = sum(t.startswith("JJ") for t in tags)
        rb_cnt  = sum(t.startswith("RB") for t in tags)
        nn_cnt  = sum(t.startswith("NN") for t in tags)

        # Dependency tree depth
        def depth(tok):
            d = 0
            while tok.head != tok:
                d += 1
                tok = tok.head
            return d
        max_dep_depth = max((depth(t) for t in doc), default=0)
        root_tag_is_vbd = int(any(t.dep_ == "ROOT" and t.tag_ == "VBD" for t in doc))
        root_tag_is_vb  = int(any(t.dep_ == "ROOT" and t.tag_ == "VB" for t in doc))

        rows.append({
            # Exact 4 replicated dataset features
            "word_count": wc,
            "char_count": cc,
            "avg_word_length": awl,
            "punctuation_density": pdens,
            # Extended stylometry
            "avg_word_len_clean": awl_clean,
            "max_word_len": max((len(w.strip(string.punctuation)) for w in words), default=0),
            "punct_cnt": punct_cnt,
            "non_ascii_cnt": sum(1 for c in s if ord(c) > 127),
            "has_curly_apos": int("’" in s),
            "has_straight_apos": int("'" in s),
            "has_comma": int("," in s),
            "comma_cnt": s.count(","),
            "has_colon_semi": int((":" in s) or (";" in s)),
            "has_excl_q": int(("!" in s) or ("?" in s)),
            "has_dash_ellip": int(("—" in s) or ("-" in s) or ("…" in s)),
            "num_sentences": len(re.findall(r"[.!?]+", s)),
            "ends_period": int(s.endswith(".")),
            "starts_The": int(s.startswith("The ")),
            "starts_A_An": int(bool(re.match(r"^(?:A|An) ", s))),
            "starts_She_He": int(bool(re.match(r"^(?:She|He) ", s))),
            "starts_I_You_We": int(bool(re.match(r"^(?:I|I'|I’|You|We) ", s))),
            "pron12_cnt": len(pron12_re.findall(s)),
            "pron3_cnt": len(pron3_re.findall(s)),
            "ed_cnt": len(ed_re.findall(s)),
            "ing_cnt": len(ing_re.findall(s)),
            "ly_cnt": len(ly_re.findall(s)),
            "tech_cnt": len(tech_re.findall(s)),
            "modal_cnt": len(modal_re.findall(s)),
            "has_scene_prep": int(bool(scene_re.search(s))),
            "upper_mid_words": sum(1 for w in words[1:] if w[0].isupper()),
            "ttr": ttr,
            "wc_out_of_ai_range": int(wc < 4 or wc > 17),
            "cc_out_of_ai_range": int(cc < 30 or cc > 101),
            # POS & Syntax
            "vbd_cnt": vbd_cnt, "vbd_ratio": vbd_cnt / ntok,
            "vbn_cnt": vbn_cnt, "vbn_ratio": vbn_cnt / ntok,
            "vbp_vbz_ratio": vbp_vbz / ntok,
            "vb_ratio": vb_cnt / ntok,
            "prp_ratio": prp_cnt / ntok,
            "dt_ratio": dt_cnt / ntok,
            "md_ratio": md_cnt / ntok,
            "jj_ratio": jj_cnt / ntok,
            "rb_ratio": rb_cnt / ntok,
            "nn_ratio": nn_cnt / ntok,
            "max_dep_depth": max_dep_depth,
            "root_is_vbd": root_tag_is_vbd,
            "root_is_vb": root_tag_is_vb,
        })
    return pd.DataFrame(rows)

if os.path.exists(feat_cache_tr) and os.path.exists(feat_cache_te):
    X_tab_tr = pd.read_parquet(feat_cache_tr)
    X_tab_te = pd.read_parquet(feat_cache_te)
    print(f"Loaded cached tabular features: {X_tab_tr.shape}")
else:
    all_texts = pd.concat([train["text"], test["text"]], ignore_index=True).tolist()
    df_lm  = extract_lm_memorization_features(all_texts, "gpt2-large")
    df_pos = extract_stylometry_and_pos(all_texts)
    X_all  = pd.concat([df_pos, df_lm], axis=1)
    X_tab_tr = X_all.iloc[:len(train)].reset_index(drop=True)
    X_tab_te = X_all.iloc[len(train):].reset_index(drop=True)
    X_tab_tr.to_parquet(feat_cache_tr)
    X_tab_te.to_parquet(feat_cache_te)
    print(f"Built & cached tabular features: {X_tab_tr.shape}")

# Train 10-Fold CatBoost & LightGBM (shallow, heavily regularized for N=1039)
oof_cb, test_cb   = np.zeros(len(train)), np.zeros(len(test))
oof_lgb, test_lgb = np.zeros(len(train)), np.zeros(len(test))

for fold in range(N_FOLDS):
    tr_idx = np.where(folds != fold)[0]
    va_idx = np.where(folds == fold)[0]

    X_tr, y_tr = X_tab_tr.iloc[tr_idx], y[tr_idx]
    X_va, y_va = X_tab_tr.iloc[va_idx], y[va_idx]

    cb = CatBoostClassifier(
        iterations=1200, learning_rate=0.03, depth=4, l2_leaf_reg=5.0,
        subsample=0.8, eval_metric="Logloss", random_seed=SEED + fold, verbose=0
    )
    cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=100, verbose=False)
    oof_cb[va_idx] = cb.predict_proba(X_va)[:, 1]
    test_cb += cb.predict_proba(X_tab_te)[:, 1] / N_FOLDS

    model_lgb = lgb.LGBMClassifier(
        n_estimators=1200, learning_rate=0.03, num_leaves=10, max_depth=4,
        min_child_samples=18, subsample=0.8, colsample_bytree=0.75,
        reg_alpha=0.5, reg_lambda=2.0, random_state=SEED + fold, verbose=-1
    )
    model_lgb.fit(
        X_tr, y_tr, eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )
    oof_lgb[va_idx] = model_lgb.predict_proba(X_va)[:, 1]
    test_lgb += model_lgb.predict_proba(X_tab_te)[:, 1] / N_FOLDS

np.save(os.path.join(OUT_DIR, "oof_catboost.npy"), oof_cb)
np.save(os.path.join(OUT_DIR, "test_catboost.npy"), test_cb)
np.save(os.path.join(OUT_DIR, "oof_lgbm.npy"), oof_lgb)
np.save(os.path.join(OUT_DIR, "test_lgbm.npy"), test_lgb)

evaluate_predictions("Stream B1: CatBoost", oof_cb)
evaluate_predictions("Stream B2: LightGBM", oof_lgb)

Extracting frozen LM memorization features with gpt2-large...


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Built & cached tabular features: (1039, 63)
[Stream B1: CatBoost       ] AUC: 0.99645 | LogLoss: 0.07718 | F1@0.50: 0.97271 | BestF1: 0.97356 (th=0.665)
[Stream B2: LightGBM       ] AUC: 0.99667 | LogLoss: 0.07726 | F1@0.50: 0.97193 | BestF1: 0.97396 (th=0.400)


(np.float64(0.9966688651909381), 0.973963355834137)

# 06. Transformer Architecture and Training Framework
### Dynamic Tokenization, Attention Pooling, and Multi-Sample Dropout

In [6]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import torch
gc.collect()
torch.cuda.empty_cache()

import bitsandbytes as bnb
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_cosine_schedule_with_warmup

class DynamicTextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128, labels=None):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            max_length=max_len,
            padding=False,
        )
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
        }
        if self.labels is not None:
            item["label"] = float(self.labels[idx])
        return item

def make_collate_fn(pad_token_id):
    def collate_fn(batch):
        max_l = max(len(x["input_ids"]) for x in batch)
        # Align to multiple of 8 for tensor-core efficiency
        max_l = ((max_l + 7) // 8) * 8
        input_ids, attention_mask = [], []
        for x in batch:
            pad_len = max_l - len(x["input_ids"])
            input_ids.append(x["input_ids"] + [pad_token_id] * pad_len)
            attention_mask.append(x["attention_mask"] + [0] * pad_len)
        out = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        }
        if "label" in batch[0]:
            out["label"] = torch.tensor([x["label"] for x in batch], dtype=torch.float32)
        return out
    return collate_fn

class MultiSampleDropoutEncoder(nn.Module):
    def __init__(self, model_name, use_grad_ckpt=True):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        if hasattr(self.config, "hidden_dropout_prob"):
            self.config.hidden_dropout_prob = 0.1
        if hasattr(self.config, "attention_probs_dropout_prob"):
            self.config.attention_probs_dropout_prob = 0.1

        self.backbone = AutoModel.from_pretrained(
            model_name,
            config=self.config,
            **dtype_kwarg(torch.float32),
        )
        if use_grad_ckpt and hasattr(self.backbone, "gradient_checkpointing_enable"):
            try:
                self.backbone.gradient_checkpointing_enable(
                    gradient_checkpointing_kwargs={"use_reentrant": False}
                )
            except TypeError:
                self.backbone.gradient_checkpointing_enable()

        hidden = self.config.hidden_size
        self.attn_pool = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.Tanh(),
            nn.Linear(hidden // 2, 1)
        )
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in [0.1, 0.2, 0.3, 0.4, 0.5]])
        self.fc = nn.Linear(hidden * 2, 1)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        h = out.last_hidden_state.float()

        mask = attention_mask.unsqueeze(-1).float()
        mean_pool = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)

        attn_w = self.attn_pool(h).masked_fill(mask == 0, -1e4)
        attn_w = torch.softmax(attn_w, dim=1)
        attn_pool = (h * attn_w).sum(dim=1)

        feat = torch.cat([mean_pool, attn_pool], dim=-1)
        logits = sum(self.fc(dp(feat)) for dp in self.dropouts) / len(self.dropouts)
        return logits.squeeze(-1)

def predict_encoder(model, loader, use_bf16):
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(DEVICE, non_blocking=True)
            mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=use_bf16):
                logits = model(ids, mask)
            preds.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(preds)

def run_10fold_encoder(
    model_name, tag, seeds=[42], lr=1.2e-5, epochs=5, batch_size=4, grad_accum=4,
    weight_decay=0.03, label_smoothing=0.02
):
    use_fast = ("deberta-v3" not in model_name.lower())
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=use_fast)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    collate_fn = make_collate_fn(pad_id)

    test_ds = DynamicTextDataset(test["text"].values, tokenizer, MAX_LEN)
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=collate_fn, pin_memory=True)

    use_bf16 = HAS_BF16 and ("deberta" not in model_name.lower())
    criterion = nn.BCEWithLogitsLoss()

    all_seed_oof  = []
    all_seed_test = []

    for seed in seeds:
        oof_path  = os.path.join(OUT_DIR, f"oof_{tag}_s{seed}.npy")
        test_path = os.path.join(OUT_DIR, f"test_{tag}_s{seed}.npy")
        done_path = os.path.join(OUT_DIR, f"done_{tag}_s{seed}.json")

        oof_preds  = np.load(oof_path)  if os.path.exists(oof_path)  else np.zeros(len(train))
        test_folds = np.load(test_path) if os.path.exists(test_path) else np.zeros((N_FOLDS, len(test)))
        done_folds = json.load(open(done_path)) if os.path.exists(done_path) else []

        for fold in range(N_FOLDS):
            if fold in done_folds:
                print(f"[{tag} | seed={seed} | fold={fold}] Already complete — skipping.")
                continue

            t0 = time.time()
            seed_everything(seed + fold)
            gc.collect()
            torch.cuda.empty_cache()

            tr_idx = np.where(folds != fold)[0]
            va_idx = np.where(folds == fold)[0]

            tr_ds = DynamicTextDataset(train["text"].values[tr_idx], tokenizer, MAX_LEN, y[tr_idx])
            va_ds = DynamicTextDataset(train["text"].values[va_idx], tokenizer, MAX_LEN, y[va_idx])
            tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, pin_memory=True)
            va_loader = DataLoader(va_ds, batch_size=32, shuffle=False, collate_fn=collate_fn, pin_memory=True)

            model = MultiSampleDropoutEncoder(model_name, use_grad_ckpt=True).to(DEVICE)
            if fold == 0:
                p_dtype = next(model.backbone.parameters()).dtype
                print(f"[{tag}] Backbone dtype: {p_dtype} | bf16: {use_bf16} | micro_bs={batch_size} x accum={grad_accum}")

            # 4 clean parameter groups (reduces optimizer overhead)
            no_decay = ("bias", "LayerNorm.weight", "norm.weight")
            bb_dec, bb_nodec, hd_dec, hd_nodec = [], [], [], []
            for n, p in model.named_parameters():
                if not p.requires_grad:
                    continue
                is_nd = any(nd in n for nd in no_decay)
                if "backbone" in n:
                    (bb_nodec if is_nd else bb_dec).append(p)
                else:
                    (hd_nodec if is_nd else hd_dec).append(p)

            param_groups = [
                {"params": bb_dec,   "lr": lr,       "weight_decay": weight_decay},
                {"params": bb_nodec, "lr": lr,       "weight_decay": 0.0},
                {"params": hd_dec,   "lr": lr * 5.0, "weight_decay": weight_decay},
                {"params": hd_nodec, "lr": lr * 5.0, "weight_decay": 0.0},
            ]
            optimizer = bnb.optim.AdamW8bit(param_groups)

            opt_steps_per_epoch = math.ceil(len(tr_loader) / grad_accum)
            total_steps = opt_steps_per_epoch * epochs
            scheduler = get_cosine_schedule_with_warmup(
                optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
            )

            eval_every = max(1, opt_steps_per_epoch // 2)
            best_score = -1e9
            best_va_preds = None
            best_state = None

            opt_step = 0
            for ep in range(epochs):
                model.train()
                optimizer.zero_grad(set_to_none=True)

                for step_i, batch in enumerate(tr_loader, 1):
                    ids = batch["input_ids"].to(DEVICE, non_blocking=True)
                    mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
                    lbl = batch["label"].to(DEVICE, non_blocking=True)
                    lbl_smooth = lbl * (1.0 - label_smoothing) + 0.5 * label_smoothing

                    with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=use_bf16):
                        logits = model(ids, mask)
                        loss = criterion(logits, lbl_smooth) / grad_accum

                    if not torch.isfinite(loss):
                        raise RuntimeError(f"Non-finite loss at ep={ep}, step={step_i}, fold={fold}")

                    loss.backward()

                    if (step_i % grad_accum == 0) or (step_i == len(tr_loader)):
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        optimizer.step()
                        optimizer.zero_grad(set_to_none=True)
                        scheduler.step()
                        opt_step += 1

                        if (opt_step % eval_every == 0) or (opt_step == total_steps):
                            va_probs = predict_encoder(model, va_loader, use_bf16)
                            if not np.isfinite(va_probs).all():
                                raise RuntimeError(f"Non-finite val probs at step={opt_step}, fold={fold}")
                            va_auc = roc_auc_score(y[va_idx], va_probs)
                            va_ll  = log_loss(y[va_idx], np.clip(va_probs, 1e-6, 1 - 1e-6))
                            score = va_auc - 0.15 * va_ll
                            if score > best_score:
                                best_score = score
                                best_va_preds = va_probs.copy()
                                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                            model.train()

            model.load_state_dict(best_state)
            te_probs = predict_encoder(model, test_loader, use_bf16)

            oof_preds[va_idx] = best_va_preds
            test_folds[fold]  = te_probs
            done_folds.append(fold)

            np.save(oof_path, oof_preds)
            np.save(test_path, test_folds)
            json.dump(done_folds, open(done_path, "w"))

            f1_f = f1_score(y[va_idx], (best_va_preds >= 0.5).astype(int))
            auc_f = roc_auc_score(y[va_idx], best_va_preds)
            peak_gb = torch.cuda.max_memory_allocated() / (1024**3)
            torch.cuda.reset_peak_memory_stats()
            print(f"[{tag} | s={seed} | fold={fold}] AUC={auc_f:.5f} | F1@0.5={f1_f:.5f} | PeakVRAM={peak_gb:.2f}GB | {time.time()-t0:.1f}s")

            del model, optimizer, scheduler, best_state
            gc.collect()
            torch.cuda.empty_cache()

        all_seed_oof.append(oof_preds)
        all_seed_test.append(test_folds.mean(axis=0))

    final_oof  = np.mean(all_seed_oof, axis=0)
    final_test = np.mean(all_seed_test, axis=0)
    np.save(os.path.join(OUT_DIR, f"oof_{tag}.npy"), final_oof)
    np.save(os.path.join(OUT_DIR, f"test_{tag}.npy"), final_test)
    evaluate_predictions(f"Encoder: {tag}", final_oof)
    return final_oof, final_test

# 07. Transformer Tokenizer Verification
### Validating DeBERTa-v3-large and ModernBERT-large Tokenizers

In [7]:
!pip -q install -U protobuf sentencepiece

from transformers import AutoTokenizer

# Verify both tokenizers load cleanly before starting 10-fold training
for name in ["microsoft/deberta-v3-large", "answerdotai/ModernBERT-large"]:
    use_fast = ("deberta-v3" not in name.lower())
    tok = AutoTokenizer.from_pretrained(name, use_fast=use_fast)
    print(f"OK: {name} -> {type(tok).__name__} | test tokens: {tok.tokenize('I’m ready—let’s go.')[:6]}")

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

OK: microsoft/deberta-v3-large -> DebertaV2Tokenizer | test tokens: ['▁I', '’', 'm', '▁ready', '—', 'let']


config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

OK: answerdotai/ModernBERT-large -> TokenizersBackend | test tokens: ['I', 'âĢĻ', 'm', 'Ġready', 'âĢĶ', 'let']


# 08. Transformer-Based Model Training
### DeBERTa-v3-large and ModernBERT-large

In [8]:
# Stream C: microsoft/deberta-v3-large (2 seeds x 10 folds)
oof_deberta, test_deberta = run_10fold_encoder(
    model_name="microsoft/deberta-v3-large",
    tag="deberta_v3_large",
    seeds=DEBERTA_SEEDS,
    lr=1.0e-5,
    epochs=5,
    batch_size=4,
    grad_accum=4,
    weight_decay=0.03,
    label_smoothing=0.02,
)

# Stream D: answerdotai/ModernBERT-large (10 folds)
oof_modernbert, test_modernbert = run_10fold_encoder(
    model_name="answerdotai/ModernBERT-large",
    tag="modernbert_large",
    seeds=[SEED],
    lr=2.0e-5,
    epochs=5,
    batch_size=4,
    grad_accum=4,
    weight_decay=0.03,
    label_smoothing=0.02,
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  874MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  874MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... (86 outputs left out)

[modernbert_large | s=42 | fold=8] AUC=0.99852 | F1@0.5=0.98039 | PeakVRAM=4.39GB | 292.4s


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[modernbert_large | s=42 | fold=9] AUC=1.00000 | F1@0.5=1.00000 | PeakVRAM=4.39GB | 288.8s
[Encoder: modernbert_large ] AUC: 0.99890 | LogLoss: 0.03684 | F1@0.50: 0.99225 | BestF1: 0.99225 (th=0.400)


# 09. Qwen 2.5 7B Fine-Tuning with QLoRA
### 4-Bit Quantization, Low-Rank Adaptation, and Cross-Validation

In [9]:
import os, gc, math, json, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import bitsandbytes as bnb
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from sklearn.metrics import f1_score, roc_auc_score, log_loss

class QLoRADataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=160, labels=None):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            max_length=max_len,
            padding=False,
        )
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
        }
        if self.labels is not None:
            item["label"] = float(self.labels[idx])
        return item

def make_qlora_collate_fn(pad_token_id):
    def collate_fn(batch):
        max_l = max(len(x["input_ids"]) for x in batch)
        max_l = ((max_l + 7) // 8) * 8
        input_ids, attention_mask = [], []
        for x in batch:
            pad_len = max_l - len(x["input_ids"])
            input_ids.append(x["input_ids"] + [pad_token_id] * pad_len)
            attention_mask.append(x["attention_mask"] + [0] * pad_len)
        out = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        }
        if "label" in batch[0]:
            out["label"] = torch.tensor([x["label"] for x in batch], dtype=torch.float32)
        return out
    return collate_fn

def format_llm_prompt(text):
    return (
        "Classify whether the following sentence is Human-written or AI-generated.\n"
        f"Sentence: {text}\nClassification:"
    )

def run_10fold_qlora(
    model_name="Qwen/Qwen2.5-7B-Instruct",
    tag="qwen25_7b_qlora",
    lr=8e-5,
    epochs=4,
    batch_size=4,
    grad_accum=4,
    max_len=160,
):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    collate_fn = make_qlora_collate_fn(tokenizer.pad_token_id)

    train_prompts = [format_llm_prompt(t) for t in train["text"].values]
    test_prompts  = [format_llm_prompt(t) for t in test["text"].values]

    test_ds = QLoRADataset(test_prompts, tokenizer, max_len)
    test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, collate_fn=collate_fn, pin_memory=True)

    # Check true hardware bf16 (Ampere+ CC >= 8); use native fp16 on Tesla T4 (CC 7.5)
    native_bf16 = torch.cuda.is_available() and (torch.cuda.get_device_capability(0)[0] >= 8)
    compute_dtype = torch.bfloat16 if native_bf16 else torch.float16
    print(f"[{tag}] Using compute_dtype={compute_dtype} (native_bf16={native_bf16})")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )

    oof_path  = os.path.join(OUT_DIR, f"oof_{tag}.npy")
    test_path = os.path.join(OUT_DIR, f"test_{tag}_folds.npy")
    done_path = os.path.join(OUT_DIR, f"done_{tag}.json")

    oof_preds  = np.load(oof_path)  if os.path.exists(oof_path)  else np.zeros(len(train))
    test_folds = np.load(test_path) if os.path.exists(test_path) else np.zeros((N_FOLDS, len(test)))
    done_folds = json.load(open(done_path)) if os.path.exists(done_path) else []

    criterion = nn.BCEWithLogitsLoss()
    scaler = torch.amp.GradScaler("cuda", enabled=(not native_bf16))

    for fold in range(N_FOLDS):
        if fold in done_folds:
            print(f"[{tag} | fold={fold}] Already complete — skipping.")
            continue

        t0 = time.time()
        seed_everything(SEED + fold)
        gc.collect()
        torch.cuda.empty_cache()

        tr_idx = np.where(folds != fold)[0]
        va_idx = np.where(folds == fold)[0]

        tr_ds = QLoRADataset([train_prompts[i] for i in tr_idx], tokenizer, max_len, y[tr_idx])
        va_ds = QLoRADataset([train_prompts[i] for i in va_idx], tokenizer, max_len, y[va_idx])
        tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, pin_memory=True)
        va_loader = DataLoader(va_ds, batch_size=16, shuffle=False, collate_fn=collate_fn, pin_memory=True)

        base_model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=1,
            quantization_config=bnb_config,
            device_map={"": 0},
            **dtype_kwarg(compute_dtype),
        )
        base_model.config.pad_token_id = tokenizer.pad_token_id
        base_model.config.use_cache = False
        base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)

        lora_cfg = LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.1,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            bias="none",
            task_type=TaskType.SEQ_CLS,
        )
     

        model = get_peft_model(base_model, lora_cfg)
        for p in model.parameters():
            if p.requires_grad:
                p.data = p.data.float()

        optimizer = bnb.optim.PagedAdamW8bit(
            [p for p in model.parameters() if p.requires_grad],
            lr=lr, weight_decay=0.02
        )
        total_steps = math.ceil(len(tr_loader) / grad_accum) * epochs
        scheduler = get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
        )

        best_score = -1e9
        best_va_preds = None
        best_te_preds = None

        def eval_qlora(loader):
            model.eval()
            out_p = []
            with torch.no_grad():
                for b in loader:
                    with torch.amp.autocast("cuda", dtype=compute_dtype):
                        logits = model(
                            input_ids=b["input_ids"].to(DEVICE, non_blocking=True),
                            attention_mask=b["attention_mask"].to(DEVICE, non_blocking=True),
                        ).logits.squeeze(-1)
                    out_p.append(torch.sigmoid(logits.float()).cpu().numpy())
            return np.concatenate(out_p)

        for ep in range(epochs):
            model.train()
            optimizer.zero_grad(set_to_none=True)
            for step_i, batch in enumerate(tr_loader, 1):
                lbl = batch["label"].to(DEVICE, non_blocking=True)
                lbl_smooth = lbl * 0.98 + 0.01

                with torch.amp.autocast("cuda", dtype=compute_dtype):
                    logits = model(
                        input_ids=batch["input_ids"].to(DEVICE, non_blocking=True),
                        attention_mask=batch["attention_mask"].to(DEVICE, non_blocking=True),
                    ).logits.squeeze(-1)
                    loss = criterion(logits, lbl_smooth) / grad_accum

                scaler.scale(loss).backward()
                if (step_i % grad_accum == 0) or (step_i == len(tr_loader)):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    scheduler.step()

            va_probs = eval_qlora(va_loader)
            va_auc = roc_auc_score(y[va_idx], va_probs)
            va_ll  = log_loss(y[va_idx], np.clip(va_probs, 1e-6, 1 - 1e-6))
            score  = va_auc - 0.15 * va_ll
            if score > best_score:
                best_score = score
                best_va_preds = va_probs.copy()
                best_te_preds = eval_qlora(test_loader)

        oof_preds[va_idx] = best_va_preds
        test_folds[fold]  = best_te_preds
        done_folds.append(fold)

        np.save(oof_path, oof_preds)
        np.save(test_path, test_folds)
        json.dump(done_folds, open(done_path, "w"))

        f1_f = f1_score(y[va_idx], (best_va_preds >= 0.5).astype(int))
        auc_f = roc_auc_score(y[va_idx], best_va_preds)
        peak_gb = torch.cuda.max_memory_allocated() / (1024**3)
        torch.cuda.reset_peak_memory_stats()
        print(f"[{tag} | fold={fold}] AUC={auc_f:.5f} | F1@0.5={f1_f:.5f} | PeakVRAM={peak_gb:.2f}GB | {time.time()-t0:.1f}s")

        del model, base_model, optimizer, scheduler
        gc.collect()
        torch.cuda.empty_cache()

    final_test = test_folds.mean(axis=0)
    np.save(os.path.join(OUT_DIR, f"test_{tag}.npy"), final_test)
    evaluate_predictions(f"QLoRA: {tag}", oof_preds)
    return oof_preds, final_test

if RUN_QLORA_7B:
    oof_qwen, test_qwen = run_10fold_qlora()

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[qwen25_7b_qlora] Using compute_dtype=torch.float16 (native_bf16=False)


... (29 outputs left out)

[qwen25_7b_qlora | fold=8] AUC=0.99741 | F1@0.5=0.98039 | PeakVRAM=9.38GB | 815.0s


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-7B-Instruct
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[qwen25_7b_qlora | fold=9] AUC=1.00000 | F1@0.5=1.00000 | PeakVRAM=9.38GB | 818.1s
[QLoRA: qwen25_7b_qlora    ] AUC: 0.99978 | LogLoss: 0.02018 | F1@0.50: 0.99710 | BestF1: 0.99710 (th=0.250)


# 10. Out-of-Fold Prediction and Ensemble Learning
### Regularized Logistic Regression Stacking and Non-Negative Logit Blending

In [10]:
from scipy.special import logit, expit
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression

# Auto-load all available streams from Drive
CANDIDATES = [
    ("tfidf_lr",         "oof_tfidf_lr.npy",         "test_tfidf_lr.npy"),
    ("tfidf_svc",        "oof_tfidf_svc.npy",        "test_tfidf_svc.npy"),
    ("catboost",         "oof_catboost.npy",         "test_catboost.npy"),
    ("lgbm",             "oof_lgbm.npy",             "test_lgbm.npy"),
    ("deberta_v3_large", "oof_deberta_v3_large.npy", "test_deberta_v3_large.npy"),
    ("modernbert_large", "oof_modernbert_large.npy", "test_modernbert_large.npy"),
    ("qwen25_7b_qlora",  "oof_qwen25_7b_qlora.npy",  "test_qwen25_7b_qlora.npy"),
]

model_names, oof_list, test_list = [], [], []
print("=== Individual 10-Fold OOF Performance ===")
for name, oof_f, test_f in CANDIDATES:
    op, tp = os.path.join(OUT_DIR, oof_f), os.path.join(OUT_DIR, test_f)
    if os.path.exists(op) and os.path.exists(tp):
        o = np.load(op)
        t = np.load(tp)
        model_names.append(name)
        oof_list.append(o)
        test_list.append(t)
        evaluate_predictions(name, o)

EPS = 1e-4
Z_oof  = np.column_stack([logit(np.clip(p, EPS, 1 - EPS)) for p in oof_list])
Z_test = np.column_stack([logit(np.clip(p, EPS, 1 - EPS)) for p in test_list])

# --- Method 1: Nested 10-Fold Regularized Log-Odds Meta-Learner ---
best_C, best_meta_ll = 0.1, 1e9
for C_val in [0.02, 0.05, 0.1, 0.25, 0.5, 1.0]:
    meta_oof_tmp = np.zeros(len(train))
    for fold in range(N_FOLDS):
        tr_idx, va_idx = np.where(folds != fold)[0], np.where(folds == fold)[0]
        clf = LogisticRegression(C=C_val, max_iter=2000)
        clf.fit(Z_oof[tr_idx], y[tr_idx])
        meta_oof_tmp[va_idx] = clf.predict_proba(Z_oof[va_idx])[:, 1]
    ll_val = log_loss(y, np.clip(meta_oof_tmp, 1e-6, 1 - 1e-6))
    if ll_val < best_meta_ll:
        best_meta_ll, best_C = ll_val, C_val

meta_oof  = np.zeros(len(train))
meta_test = np.zeros(len(test))
coefs = []
for fold in range(N_FOLDS):
    tr_idx, va_idx = np.where(folds != fold)[0], np.where(folds == fold)[0]
    clf = LogisticRegression(C=best_C, max_iter=2000)
    clf.fit(Z_oof[tr_idx], y[tr_idx])
    meta_oof[va_idx] = clf.predict_proba(Z_oof[va_idx])[:, 1]
    meta_test += clf.predict_proba(Z_test)[:, 1] / N_FOLDS
    coefs.append(clf.coef_[0])


blend_oof  = np.zeros(len(train))
blend_test = np.zeros(len(test))
w_folds = []
M = len(model_names)
for fold in range(N_FOLDS):
    tr_idx, va_idx = np.where(folds != fold)[0], np.where(folds == fold)[0]
    def loss_fn(w):
        w = np.maximum(w, 0)
        w = w / (w.sum() + 1e-12)
        p = expit(Z_oof[tr_idx] @ w)
        return log_loss(y[tr_idx], np.clip(p, 1e-6, 1 - 1e-6))
    res = minimize(loss_fn, np.ones(M) / M, bounds=[(0, 1)] * M, method="SLSQP",
                   constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1.0})
    w_opt = res.x / res.x.sum()
    w_folds.append(w_opt)
    blend_oof[va_idx] = expit(Z_oof[va_idx] @ w_opt)
    blend_test += expit(Z_test @ w_opt) / N_FOLDS

print("\n=== Ensemble Results (Leak-Free Nested 10-Fold CV) ===")
auc_m, f1_m = evaluate_predictions(f"L2 Logit Stack (C={best_C})", meta_oof)
auc_b, f1_b = evaluate_predictions("Non-Neg Logit Blend", blend_oof)

print("\nAverage Non-Negative Blend Weights:")
for name, w in zip(model_names, np.mean(w_folds, axis=0)):
    print(f"  {name:22s}: {w:.4f}")

# Pick winner by LogLoss + AUC stability
ll_m = log_loss(y, np.clip(meta_oof, 1e-6, 1 - 1e-6))
ll_b = log_loss(y, np.clip(blend_oof, 1e-6, 1 - 1e-6))
if ll_b <= ll_m:
    FINAL_OOF, FINAL_TEST, WINNER = blend_oof, blend_test, "Non-Neg Logit Blend"
else:
    FINAL_OOF, FINAL_TEST, WINNER = meta_oof, meta_test, f"L2 Logit Stack (C={best_C})"
print(f"\n>>> Selected Final Ensemble: {WINNER}")


rng = np.random.default_rng(SEED)
ths = np.linspace(0.30, 0.70, 81)
boot_ths = []
for _ in range(300):
    idx = rng.integers(0, len(y), len(y))
    yb, pb = y[idx], FINAL_OOF[idx]
    best_t = max(ths, key=lambda t: f1_score(yb, (pb >= t).astype(int)))
    boot_ths.append(best_t)

BEST_TH = float(np.median(boot_ths))
final_oof_preds = (FINAL_OOF >= BEST_TH).astype(int)
print(f"Bootstrap-Median F1 Threshold: {BEST_TH:.3f}")
print(f"Final Nested OOF F1:           {f1_score(y, final_oof_preds):.5f}")
print(f"Final Nested OOF AUC:          {roc_auc_score(y, FINAL_OOF):.5f}")
print(f"Total Remaining OOF Errors:    {(final_oof_preds != y).sum()} / {len(y)}")


err = train.copy()
err["prob_ai"] = FINAL_OOF
err["pred"] = final_oof_preds
err = err[err["pred"] != err["target"]].sort_values("prob_ai")
print("\n--- Remaining OOF Errors (Audit) ---")
for _, r in err.iterrows():
    print(f"  True={r['target']} | Pred={r['pred']} | p={r['prob_ai']:.4f} | {r['text']}")

=== Individual 10-Fold OOF Performance ===
[tfidf_lr                  ] AUC: 0.99027 | LogLoss: 0.15586 | F1@0.50: 0.96538 | BestF1: 0.96724 (th=0.505)
[tfidf_svc                 ] AUC: 0.99028 | LogLoss: 0.10883 | F1@0.50: 0.96625 | BestF1: 0.96638 (th=0.350)
[catboost                  ] AUC: 0.99645 | LogLoss: 0.07718 | F1@0.50: 0.97271 | BestF1: 0.97356 (th=0.665)
[lgbm                      ] AUC: 0.99667 | LogLoss: 0.07726 | F1@0.50: 0.97193 | BestF1: 0.97396 (th=0.400)
[deberta_v3_large          ] AUC: 0.99987 | LogLoss: 0.02194 | F1@0.50: 0.99226 | BestF1: 0.99325 (th=0.340)
[modernbert_large          ] AUC: 0.99890 | LogLoss: 0.03684 | F1@0.50: 0.99225 | BestF1: 0.99225 (th=0.400)
[qwen25_7b_qlora           ] AUC: 0.99978 | LogLoss: 0.02018 | F1@0.50: 0.99710 | BestF1: 0.99710 (th=0.250)

=== Ensemble Results (Leak-Free Nested 10-Fold CV) ===
[L2 Logit Stack (C=0.1)    ] AUC: 0.99989 | LogLoss: 0.01332 | F1@0.50: 0.99613 | BestF1: 0.99613 (th=0.250)
[Non-Neg Logit Blend       ] 

# 11. Threshold Optimization and Error Analysis
### Bootstrap-Based Threshold Selection and Misclassification Inspection

In [11]:
sub = pd.DataFrame({
    "id": test["id"].values,
    "label": (FINAL_TEST >= BEST_TH).astype(int)
})

# sub_path = os.path.join(OUT_DIR, "submission.csv")
sub_path = "/kaggle/working/submission.csv"
sub.to_csv(sub_path, index=False)

# 13. Submission Validation
### Competition Format, ID Integrity, and Final Checks

In [12]:
assert list(sub.columns) == ["id", "label"], f"Bad columns: {sub.columns}"
assert len(sub) == len(test), f"Row count mismatch: {len(sub)} vs {len(test)}"
assert (sub["id"].values == test["id"].values).all(), "Test IDs do not match exactly!"
assert set(sub["label"].unique()).issubset({0, 1}), f"Invalid labels: {sub['label'].unique()}"
assert sub.isnull().sum().sum() == 0, "NaNs found in submission!"
file_mb = os.path.getsize(sub_path) / (1024 * 1024)
assert file_mb < 10.0, f"File too large: {file_mb:.2f} MB"

print(f"Saved validated submission to: {sub_path} ({file_mb:.4f} MB)")
print(f"Predicted class counts:\n{sub['label'].value_counts().to_dict()}")
print(f"Predicted AI ratio: {sub['label'].mean():.4f} (Expected i.i.d. prior: ~0.48 - 0.52)")
display(sub.head(10))

Saved validated submission to: /kaggle/working/submission.csv (0.0014 MB)
Predicted class counts:
{0: 130, 1: 130}
Predicted AI ratio: 0.5000 (Expected i.i.d. prior: ~0.48 - 0.52)


,id,label
0,1,0
1,2,1
2,3,0
3,4,1
4,5,0
5,6,1
6,7,1
7,8,0
8,9,1
9,10,1
